# TFM — Detección de Anomalías en ECG con Big Data
## Carga de Resultados en Snowflake (Anexo E)
**Andrea Ríos López — Master Big Data & Advanced Analytics**

Este notebook implementa la capa analítica del TFM: carga los resultados de los modelos ML
desde AWS S3 hacia Snowflake, completando el pipeline Bronze → Silver → Gold → Snowflake.

**Tablas cargadas:**
- `ANOMALY_SCORES` — scores de anomalía por registro y modelo
- `ECG_METADATA` — metadatos de los registros ECG del dataset PTB-XL
- `MODEL_REGISTRY` — versiones y métricas de los modelos entrenados
- `ALERT_HISTORY` — poblada por el sistema SNS (Anexo D)

---
## E.1 — Instalación de dependencias

In [ ]:
%pip install snowflake-connector-python pandas pyarrow torch wfdb --quiet

In [ ]:
dbutils.library.restartPython()

## E.2 — Configuración: librerías, S3 y Snowflake

In [ ]:
import boto3, pandas as pd, numpy as np, io, pickle, ast, os, wfdb
import torch
import torch.nn as nn
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from datetime import date

# ── Credenciales AWS S3 ──────────────────────────────────
ACCESS_KEY = ''  # completar antes de ejecutar
SECRET_KEY = ''  # completar antes de ejecutar
BUCKET     = 'ptbxl-tfm-andrea'
REGION     = 'eu-west-1'
PREFIX     = 'bronze/FileYear=2026/FileMonth=04/FileDay=13'

s3 = boto3.client('s3',
    aws_access_key_id     = ACCESS_KEY,
    aws_secret_access_key = SECRET_KEY,
    region_name           = REGION
)

# ── Conexión Snowflake ───────────────────────────────────
conn = snowflake.connector.connect(
    user      = 'RIOSLOPEZANDREA',
    password  = '',   # completar antes de ejecutar
    account   = 'elifweh-rg90787',
    warehouse = 'TFM_WH',
    database  = 'TFM_ECG',
    schema    = 'ANOMALY_DETECTION',
    role      = 'ACCOUNTADMIN'
)

cursor = conn.cursor()
cursor.execute('SELECT CURRENT_VERSION()')
print(f'S3 OK — bucket: {BUCKET}')
print(f'Snowflake OK — version: {cursor.fetchone()[0]}')

## E.3 — Carga de metadatos PTB-XL

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key=f'{PREFIX}/ptbxl_database.csv')
df_meta = pd.read_csv(io.BytesIO(obj['Body'].read()))

def diagnostico_principal(scp_str):
    try:
        d = ast.literal_eval(scp_str)
        return max(d, key=d.get) if d else 'UNKNOWN'
    except:
        return 'UNKNOWN'

df_meta['diagnostico'] = df_meta['scp_codes'].apply(diagnostico_principal)
df_patologico = df_meta[df_meta['diagnostico'] != 'NORM'].reset_index(drop=True)

print(f'Metadatos cargados: {len(df_meta):,} registros')
print(f'  - Normales:    {(df_meta["diagnostico"] == "NORM").sum():,}')
print(f'  - Patologicos: {len(df_patologico):,}')

## E.4 — Carga de modelos ML desde S3

In [ ]:
from sklearn.preprocessing import StandardScaler

# Isolation Forest
obj = s3.get_object(Bucket=BUCKET, Key='gold/models/isolation_forest_v1.pkl')
model_data = pickle.loads(obj['Body'].read())
modelo_if  = model_data['modelo']
scaler     = model_data['scaler']

# Features patológicos
obj = s3.get_object(Bucket=BUCKET,
    Key='gold/FileYear=2026/FileMonth=04/FileDay=13/features_patologicos_completo.parquet')
df_pat       = pd.read_parquet(io.BytesIO(obj['Body'].read()))
X_pat_scaled = scaler.transform(df_pat.drop(columns=['ecg_id']).values)
scores_pat   = modelo_if.decision_function(X_pat_scaled)
pred_pat     = modelo_if.predict(X_pat_scaled)
threshold_if = float(np.percentile(-scores_pat, 5))

print(f'Isolation Forest OK — {len(df_pat):,} registros patologicos')
print(f'Umbral IF: {threshold_if:.4f}')

# LSTM Autoencoder
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, latent_size=32, num_layers=2):
        super(LSTMAutoencoder, self).__init__()
        self.encoder = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                               num_layers=num_layers, batch_first=True, dropout=0.2)
        self.latent        = nn.Linear(hidden_size, latent_size)
        self.decoder_input = nn.Linear(latent_size, hidden_size)
        self.decoder = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size,
                               num_layers=num_layers, batch_first=True, dropout=0.2)
        self.output_layer  = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        _, (hidden, _) = self.encoder(x)
        encoded    = self.latent(hidden[-1])
        dec_input  = self.decoder_input(encoded).unsqueeze(1).repeat(1, seq_len, 1)
        decoded, _ = self.decoder(dec_input)
        return self.output_layer(decoded), encoded

device = torch.device('cpu')
modelo_lstm = LSTMAutoencoder().to(device)
obj = s3.get_object(Bucket=BUCKET, Key='gold/models/lstm_autoencoder_v1.pkl')
model_data2 = pickle.loads(obj['Body'].read())
modelo_lstm.load_state_dict(model_data2['modelo_state'])
modelo_lstm.eval()

print('LSTM Autoencoder OK')

## E.5 — Cálculo de errores de reconstrucción LSTM

In [ ]:
def calcular_errores(señales_array, modelo, batch_size=64):
    """Calcula el error de reconstrucción MSE por señal."""
    errores = []
    modelo.eval()
    with torch.no_grad():
        for i in range(0, len(señales_array), batch_size):
            batch = torch.FloatTensor(señales_array[i:i+batch_size]).unsqueeze(-1)
            output, _ = modelo(batch)
            mse = ((output - batch) ** 2).mean(dim=(1, 2))
            errores.extend(mse.numpy().tolist())
    return np.array(errores)

# Cargar señales desde S3 (guardadas previamente)
obj = s3.get_object(Bucket=BUCKET, Key='gold/lstm/X_train_señales.npy')
X_train = np.load(io.BytesIO(obj['Body'].read()))

obj = s3.get_object(Bucket=BUCKET, Key='gold/lstm/X_pat_señales.npy')
X_pat_señales = np.load(io.BytesIO(obj['Body'].read()))

errores_normales = calcular_errores(X_train, modelo_lstm)
errores_pat_lstm = calcular_errores(X_pat_señales, modelo_lstm)
threshold_lstm   = float(np.percentile(errores_normales, 95))

print(f'Umbral LSTM (percentil 95 normales): {threshold_lstm:.6f}')
print(f'Error medio normales:                {errores_normales.mean():.6f}')
print(f'Error medio patologicos:             {errores_pat_lstm.mean():.6f}')

## E.6 — Construcción del DataFrame `anomaly_scores`

In [ ]:
INFERENCE_DATE = str(date.today())
rows = []

# Scores Isolation Forest
for i, row in df_patologico.iterrows():
    if i < len(scores_pat):
        rows.append({
            'ecg_id'        : int(row['ecg_id']),
            'model_name'    : 'isolation_forest_v1',
            'anomaly_score' : float(-scores_pat[i]),
            'is_anomaly'    : bool(pred_pat[i] == -1),
            'threshold'     : threshold_if,
            'inference_date': INFERENCE_DATE
        })

# Errores LSTM
for i, row in df_patologico.iterrows():
    if i < len(errores_pat_lstm):
        rows.append({
            'ecg_id'        : int(row['ecg_id']),
            'model_name'    : 'lstm_autoencoder_v1',
            'anomaly_score' : float(errores_pat_lstm[i]),
            'is_anomaly'    : bool(errores_pat_lstm[i] > threshold_lstm),
            'threshold'     : threshold_lstm,
            'inference_date': INFERENCE_DATE
        })

df_anomaly_scores = pd.DataFrame(rows)
df_anomaly_scores.columns = [c.upper() for c in df_anomaly_scores.columns]
df_anomaly_scores['IS_ANOMALY']     = df_anomaly_scores['IS_ANOMALY'].astype(bool)
df_anomaly_scores['INFERENCE_DATE'] = pd.to_datetime(df_anomaly_scores['INFERENCE_DATE']).dt.date

print(f'anomaly_scores construido: {len(df_anomaly_scores):,} filas')
print(f'  - Isolation Forest: {(df_anomaly_scores["MODEL_NAME"] == "isolation_forest_v1").sum():,}')
print(f'  - LSTM Autoencoder: {(df_anomaly_scores["MODEL_NAME"] == "lstm_autoencoder_v1").sum():,}')
display(df_anomaly_scores.head())

## E.7 — Carga en Snowflake

In [ ]:
# ANOMALY_SCORES
success, _, nrows, _ = write_pandas(conn, df_anomaly_scores, 'ANOMALY_SCORES')
print(f'ANOMALY_SCORES: {nrows:,} filas — OK: {success}')

# MODEL_REGISTRY
df_registry = pd.DataFrame([
    {
        'MODEL_NAME': 'isolation_forest_v1',
        'VERSION'   : 'v1',
        'TRAIN_DATE': INFERENCE_DATE,
        'AUC_ROC'   : 0.7477,
        'AUC_PR'    : 0.8053,
        'N_RECORDS' : len(df_pat)
    },
    {
        'MODEL_NAME': 'lstm_autoencoder_v1',
        'VERSION'   : 'v1',
        'TRAIN_DATE': INFERENCE_DATE,
        'AUC_ROC'   : 0.7321,
        'AUC_PR'    : 0.7894,
        'N_RECORDS' : int(X_pat_señales.shape[0])
    }
])
df_registry['TRAIN_DATE'] = pd.to_datetime(df_registry['TRAIN_DATE']).dt.date
success, _, nrows, _ = write_pandas(conn, df_registry, 'MODEL_REGISTRY')
print(f'MODEL_REGISTRY: {nrows:,} filas — OK: {success}')

# ECG_METADATA
df_ecg_metadata = df_meta[['ecg_id', 'patient_id', 'recording_date', 'diagnostico', 'strat_fold']].copy()
df_ecg_metadata = df_ecg_metadata.rename(columns={'diagnostico': 'diagnosis'})
df_ecg_metadata.columns = [c.upper() for c in df_ecg_metadata.columns]
df_ecg_metadata['RECORDING_DATE'] = pd.to_datetime(
    df_ecg_metadata['RECORDING_DATE'], errors='coerce'
).dt.date
success, _, nrows, _ = write_pandas(conn, df_ecg_metadata, 'ECG_METADATA')
print(f'ECG_METADATA:   {nrows:,} filas — OK: {success}')

conn.close()
print('Conexion cerrada')

## E.8 — Verificación de carga

In [ ]:
# Reconectar para verificar
conn_verify = snowflake.connector.connect(
    user      = 'RIOSLOPEZANDREA',
    password  = '',   # completar
    account   = 'elifweh-rg90787',
    warehouse = 'TFM_WH',
    database  = 'TFM_ECG',
    schema    = 'ANOMALY_DETECTION',
    role      = 'ACCOUNTADMIN'
)
cur = conn_verify.cursor()

for tabla in ['ANOMALY_SCORES', 'MODEL_REGISTRY', 'ECG_METADATA', 'ALERT_HISTORY']:
    cur.execute(f'SELECT COUNT(*) FROM {tabla}')
    n = cur.fetchone()[0]
    print(f'{tabla:20s}: {n:,} filas')

cur.close()
conn_verify.close()
print('Verificacion completada')